# Day4 · 블록2 · 이미지 필터링과 엣지 검출 실습

> **강의자료**: `강의자료/04.02.OpenCV-Canny-Sobel.md`

| Part | 주제 |
|------|------|
| Part 1 | 이미지 필터링 기초 (블러) |
| Part 2 | 엣지 검출 (Sobel · Canny · Laplacian) |
| Part 3 | 퀴즈 & 복습 |

**실습 목표**
- 이미지 컨볼루션(필터링)의 개념을 이해하고 OpenCV로 직접 적용한다.
- 평균·가우시안·미디안 블러를 통해 노이즈 특성에 따른 필터 선택법을 익힌다.
- Sobel, Canny, Laplacian 엣지 검출기의 원리와 파라미터를 비교 실험한다.
- 로봇 비전에서 '전처리 → 엣지 검출 → 객체화' 파이프라인을 구현한다.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request

In [ ]:
# 인터넷에서 이미지 다운로드
def download_image(image_url, image_path):
    req = urllib.request.Request(image_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as res, open(image_path, 'wb') as f:
        f.write(res.read())
    print(f"이미지 다운로드 완료: {image_path}")

In [ ]:
image_url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/baboon.jpg"
image_path = 'baboon.jpg'
download_image(image_url, image_path)

In [ ]:
# 노이즈 추가 (블러 필터 실습용)
image = cv2.imread(image_path)
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

In [ ]:
# 가우시안 노이즈 (mean=0, std=30)
gaussian_noise = np.random.normal(0, 30, image.shape).astype(np.int16)
image_gaussian = np.clip(image.astype(np.int16) + gaussian_noise, 0, 255).astype(np.uint8)
cv2.imwrite('baboon_gaussian_noise.jpg', image_gaussian)

plt.imshow(cv2.cvtColor(image_gaussian, cv2.COLOR_BGR2RGB))

In [ ]:
# 소금-후추(Salt-and-Pepper) 노이즈
image_sp = image.copy()
num_pixels = int(image.size * 0.02)  # 전체 픽셀의 2%
coords = [np.random.randint(0, i, num_pixels) for i in image.shape[:2]]
image_sp[coords[0], coords[1]] = 255  # 소금(흰색)
coords = [np.random.randint(0, i, num_pixels) for i in image.shape[:2]]
image_sp[coords[0], coords[1]] = 0    # 후추(검정)
cv2.imwrite('fbaboon_sp_noise.jpg', image_sp)

plt.imshow(cv2.cvtColor(image_sp, cv2.COLOR_BGR2RGB))

---
## Part 1: 이미지 필터링 기초

로봇 카메라 이미지에서 노이즈를 제거하는 세 가지 대표 블러 필터를 실습합니다.

### 1.1 필터링(컨볼루션)이란 무엇인가?

- **필터링**: 이미지의 특정 특징을 강조하거나 원치 않는 부분을 제거하는 과정
- **직관적 이해**: '주변을 살펴보고 현재 내 값을 결정하기'
    - 각 픽셀을 하나씩 지나가면서 해당 픽셀과 주변 픽셀들의 값을 특정 규칙으로 섞어 새로운 값을 계산
- **컨볼루션(Convolution)**: 이미지를 돋보기로 훑으며 색을 부드럽게 섞거나 테두리를 찾아내는 과정
- **로봇 비전에서의 역할**: 노이즈 제거 → 엣지 검출 → 물체 인식으로 이어지는 **파이프라인의 첫 단계**

### 1.2 커널(Kernel)의 역할

- **커널**: '주변 값을 어떻게 섞을 것인가'에 대한 규칙서
    - 보통 3×3이나 5×5 크기의 작은 격자 모양 행렬(Matrix)
    - 이 격자가 이미지를 픽셀 단위로 이동하며 계산을 수행
- **커널 크기의 영향**
    - 커질수록 더 넓은 범위 참고 → 이미지가 더 많이 흐려짐
    - **반드시 홀수(3, 5, 7...)로 설정** → 중앙 픽셀이 명확히 정의됨

### 1.3 대표적인 노이즈 제거 필터 종류 비교

| 필터 | 동작 원리 | 장점 | 단점 |
|------|-----------|------|------|
| **평균 블러** | 커널 영역 픽셀의 산술 평균 | 단순, 빠름 | 엣지까지 흐려짐 |
| **가우시안 블러** | 거리에 따라 가중 평균 | 자연스럽고 구조 유지 | 계산량 다소 증가 |
| **미디안 블러** | 주변 픽셀 정렬 후 중간값 | 점 노이즈 제거에 탁월 | 계산량 증가 |

**선택 가이드**
- 일반 노이즈 제거 → **가우시안 블러** (엣지 검출 전처리 표준)
- 소금-후추(Salt-and-Pepper) 노이즈 → **미디안 블러**
- 빠른 처리가 최우선 → **평균 블러**

### 1.4 세 가지 블러 필터 코드 예시

```python
# 평균 블러: 5x5 크기 커널 사용
blur_avg = cv2.blur(gray, (5, 5))

# 가우시안 블러: ksize=(5, 5), sigmaX=0 (자동 계산)
blur_gaussian = cv2.GaussianBlur(gray, (5, 5), 0)

# 미디안 블러: 커널 크기는 숫자 하나(홀수)만 전달
blur_median = cv2.medianBlur(gray, 5)
```

**참고**
https://pyimagesearch.com/2021/04/28/opencv-smoothing-and-blurring/

### 1.5 (실습) 로봇 카메라 이미지 노이즈 제거

로봇 카메라로 수신한 이미지에 세 가지 블러 필터를 각각 적용하고 결과를 비교합니다.

- `baboon_gaussian_noise.jpg` 파일을 불러와 그레이스케일로 변환합니다.
- 평균 블러, 가우시안 블러, 미디안 블러를 적용하고 결과를 화면에 출력합니다.

In [ ]:
# 1. 이미지 로드 (로봇 카메라로부터 수신했다고 가정)
image = cv2.imread('baboon_gaussian_noise.jpg')

# 2. 그레이스케일 변환 (계산량 감소 및 구조 분석 용이)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

In [ ]:
plt.imshow(gray, cmap="gray")

In [ ]:
# 3. 다양한 필터 적용 실습
blur_avg      = cv2.blur(gray, (5, 5))             # (A) 평균 블러
# cv2.imshow('Blur Avg', blur_avg)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
plt.imshow(blur_avg, cmap="gray")

In [ ]:
# [이해를 위한 직접 구현] 평균 블러 (Average Blur) — cv2.blur() 동작 원리
#
# 원리: 픽셀 (y, x) 를 중심으로 k×k 이웃 픽셀 값의 평균으로 대체
#   출력[y, x] = (이웃 픽셀 합) / (k * k)
#
# 예) k=5 일 때 5×5=25개 픽셀의 평균 → 고주파 노이즈가 희석(스무딩)됨

def apply_kernel(img, kernel):
    """커널을 이미지에 적용하는 공통 함수 — 커널만 바꾸면 다양한 필터로 활용 가능"""
    h, w = img.shape
    k = kernel.shape[0]
    pad = k // 2
    padded = np.pad(img, pad, mode='reflect').astype(np.float64)  # cv2.blur 기본값
    out = np.zeros_like(img, dtype=np.float64)

    for y in range(h):
        for x in range(w):
            region = padded[y : y + k, x : x + k]
            out[y, x] = (region * kernel).sum()

    return np.round(out).astype(np.uint8)

In [ ]:
def my_average_blur(img, k=5):
    kernel = np.ones((k, k), dtype=np.float32) / (k * k)   # 합이 1인 균일 커널
    return apply_kernel(img, kernel)

blur_avg_manual = my_average_blur(gray, k=5)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(blur_avg,        cmap="gray"); axes[0].set_title("cv2.blur (5×5)")
axes[1].imshow(blur_avg_manual, cmap="gray"); axes[1].set_title("my_average_blur (5×5)")
plt.tight_layout()
plt.show()

print("최대 픽셀 오차:", np.max(np.abs(blur_avg.astype(int) - blur_avg_manual.astype(int))))

In [ ]:
blur_gaussian = cv2.GaussianBlur(gray, (5, 5), 1.1)  # (B) 가우시안 블러
# cv2.imshow('Blur Gaussian', blur_gaussian)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
plt.imshow(blur_gaussian, cmap="gray")

In [ ]:
# [이해를 위한 직접 구현] 가우시안 블러 (Gaussian Blur) — cv2.GaussianBlur() 동작 원리
#
# 원리: 중심에서 멀수록 가중치가 작아지는 정규분포(가우시안) 커널로 가중 평균
#   G(x,y) = (1/(2π σ²)) * exp(-(x²+y²) / (2σ²))
#
# sigma=0 이면 OpenCV는 sigma = 0.3*(k//2 - 1) + 0.8 공식으로 자동 계산
# 커널 값 합 = 1 로 정규화 → 전체 밝기가 보존됨

def gaussian_kernel(k, sigma):
    ax = np.arange(k) - k // 2          # [-2, -1, 0, 1, 2] (k=5)
    xx, yy = np.meshgrid(ax, ax)        # 2D 격자 좌표
    kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return kernel / kernel.sum()        # 합이 1이 되도록 정규화

def my_gaussian_blur(img, k=5, sigma=0):
    if sigma == 0:
        sigma = 0.3 * (k // 2 - 1) + 0.8

    kernel = gaussian_kernel(k, sigma)
    return apply_kernel(img, kernel)

blur_gaussian_manual = my_gaussian_blur(gray, k=5, sigma=1.1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(blur_gaussian,        cmap="gray"); axes[0].set_title("cv2.GaussianBlur (5×5)")
axes[1].imshow(blur_gaussian_manual, cmap="gray"); axes[1].set_title("my_gaussian_blur (5×5)")
plt.tight_layout()
plt.show()

print("최대 픽셀 오차:", np.max(np.abs(blur_gaussian.astype(int) - blur_gaussian_manual.astype(int))))

In [ ]:
blur_median   = cv2.medianBlur(gray, 5)            # (C) 미디안 블러
# cv2.imshow('Blur Median', blur_median)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
plt.imshow(blur_median, cmap="gray")

In [ ]:
# [이해를 위한 직접 구현] 미디안 블러 (Median Blur) — cv2.medianBlur() 동작 원리
#
# 원리: k×k 이웃 픽셀을 크기 순으로 정렬한 뒤 중간값(median)으로 대체
#   평균 대신 중간값을 사용하므로 소금-후추 노이즈(극단값)가 결과에 영향을 주지 않음
#
# 예) [0, 120, 122, 118, 255] → 정렬 후 중간값 = 120
#      ↑ 소금(255) · 후추(0) 노이즈가 자연스럽게 제거됨

def my_median_blur(img, k=5):
    h, w = img.shape
    pad = k // 2
    padded = np.pad(img, pad, mode='edge')
    out = np.zeros_like(img, dtype=np.uint8)

    for y in range(h):
        for x in range(w):
            region = padded[y : y + k, x : x + k]
            out[y, x] = np.median(region)   # 정렬 후 중간값 — 극단값 노이즈를 무력화

    return out

blur_median_manual = my_median_blur(gray, k=5)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(blur_median,        cmap="gray"); axes[0].set_title("cv2.medianBlur (5)")
axes[1].imshow(blur_median_manual, cmap="gray"); axes[1].set_title("my_median_blur (5)")
plt.tight_layout()
plt.show()

print("최대 픽셀 오차:", np.max(np.abs(blur_median.astype(int) - blur_median_manual.astype(int))))

In [ ]:
blur_bilateral = cv2.bilateralFilter(gray, d=9, sigmaColor=75, sigmaSpace=75)  # (D) 바이래터럴 필터
# cv2.imshow('Blur Bilateral', blur_median)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
plt.imshow(blur_bilateral, cmap="gray")

In [ ]:
# [이해를 위한 직접 구현] 바이래터럴 필터 (Bilateral Filter) — cv2.bilateralFilter() 동작 원리
#
# 원리: 공간 거리(space)와 밝기 차이(color) 두 가지 가중치를 동시에 적용
#   w(y,x) = exp(-거리² / (2·σ_space²)) × exp(-밝기차² / (2·σ_color²))
#
# 가우시안 블러와의 차이:
#   가우시안: 거리만 고려 → 엣지도 흐려짐
#   바이래터럴: 거리 + 밝기 차이 고려 → 밝기가 비슷한 이웃만 평균 → 엣지 보존

def my_bilateral_filter(img, d=9, sigma_color=75, sigma_space=75):
    h, w = img.shape
    k = d
    pad = k // 2
    padded = np.pad(img, pad, mode='reflect').astype(np.float64)
    out = np.zeros_like(img, dtype=np.float64)

    # 공간 가중치 커널은 미리 계산 (모든 픽셀에서 동일)
    ax = np.arange(k) - k // 2
    xx, yy = np.meshgrid(ax, ax)
    space_kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma_space**2))

    # OpenCV와 동일하게 중심으로부터 거리가 반지름(d//2)보다 큰 곳의 가중치는 0으로 제외 (원형 마스크)
    radius = k // 2
    distance = np.sqrt(xx**2 + yy**2)
    space_kernel[distance > radius] = 0

    for y in range(h):
        for x in range(w):
            region = padded[y : y + k, x : x + k]
            center = padded[y + pad, x + pad]

            # 밝기 차이 가중치: 중심 픽셀과 색상 차이가 클수록 가중치 감소 → 엣지 보존
            color_kernel = np.exp(-((region - center)**2) / (2 * sigma_color**2))

            combined = space_kernel * color_kernel   # 두 가중치 결합
            out[y, x] = (region * combined).sum() / combined.sum()   # 정규화된 가중 평균

    return np.round(out).astype(np.uint8)

blur_bilateral_manual = my_bilateral_filter(gray, d=9, sigma_color=75, sigma_space=75)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(blur_bilateral,        cmap="gray"); axes[0].set_title("cv2.bilateralFilter (d=9)")
axes[1].imshow(blur_bilateral_manual, cmap="gray"); axes[1].set_title("my_bilateral_filter (d=9)")
plt.tight_layout()
plt.show()

print("최대 픽셀 오차:", np.max(np.abs(blur_bilateral.astype(int) - blur_bilateral_manual.astype(int))))

### 1.6 커널 크기 선택 기준 및 흔한 실수

**커널 크기 선택 기준**
- 작은 노이즈 제거: 3×3 또는 5×5 작은 커널
- 큰 형태만 남기고 싶을 때: 7×7 이상 큰 커널
- **주의: 커널 크기는 반드시 홀수로 설정** (중앙 픽셀 명확히 정의)

**입문자가 자주 하는 실수 3가지**
1. **커널 크기 과다 설정** — 가장 작은 크기부터 조금씩 늘려가는 것이 좋음
2. **노이즈 종류와 필터의 미스매치** — 점 형태 노이즈에는 미디안 블러가 효과적
3. **엣지 손실 방치** — 테두리를 지키며 노이즈만 제거하려면 바이래터럴 필터(Bilateral Filter) 활용

### 1.7 (TODO) 커널 크기 변화 실험

가우시안 블러의 커널 크기를 3×3, 7×7, 15×15로 바꿔가며 흐림 정도 변화를 관찰하세요.

In [ ]:
# TODO: 여기에 구현하세요
# 힌트:
#   - 위의 gray 이미지를 사용합니다.
#   - 커널 크기를 (3,3), (7,7), (15,15)로 바꿔가며 GaussianBlur를 각각 적용하세요.
#   - 각 결과를 plt.imshow()로 출력하고 차이를 비교하세요.

---
## Part 2: 엣지 검출 — Sobel · Canny · Laplacian

로봇 비전에서 차선, 장애물, 물체의 경계를 찾는 핵심 알고리즘을 실습합니다.

### 2.1 엣지(Edge)란 무엇인가?

- **엣지**: 이미지 내에서 밝기(강도)가 급격하게 변하는 지점
    - 보통 객체와 배경 사이의 **경계선** 또는 서로 다른 객체 간의 **테두리**

**로봇 비전에서 중요한 이유**
- **객체 인식 및 위치 추적**: 차선, 장애물, 도구 등을 인식하는 기초 데이터
- **데이터 경량화**: 색상·텍스처 정보 대신 윤곽선만 사용 → 계산 복잡도 감소
- **공간 지각**: 객체의 크기와 방향 파악 → 충돌 방지 및 경로 계획

### 2.2 Sobel(소벨) 필터 — 원리와 방향

- 이미지 강도의 **그래디언트(Gradient, 기울기)를 계산하여 엣지를 찾는 1차 미분 기반 연산자**

**X방향 Sobel** — 수직 방향 엣지 검출 (수평 방향 밝기 변화 감지)
```
Gx = [-1  0  1]
      [-2  0  2]
      [-1  0  1]
```

**Y방향 Sobel** — 수평 방향 엣지 검출 (수직 방향 밝기 변화 감지)
```
Gy = [-1 -2 -1]
      [ 0  0  0]
      [ 1  2  1]
```

**두 방향 합치기 (Magnitude)**: G = sqrt(Gx² + Gy²)

**주요 파라미터**
- `ksize`: 소벨 커널의 크기 (보통 1, 3, 5, 7 등 홀수)
- `dx`, `dy`: 미분 방향 (`dx=1, dy=0` → X방향 / `dx=0, dy=1` → Y방향)

**한계점**
- 노이즈에 매우 취약
- 엣지가 두껍게 검출되는 경향 → 정밀한 윤곽선 추출에 한계

**참고**
https://automaticaddison.com/how-the-sobel-operator-works/

In [ ]:
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6b/Road_in_Norway.jpg/1280px-Road_in_Norway.jpg"
image_path = 'road_image.jpg'
download_image(image_url, image_path)

In [ ]:
# 이미지 로드
image = cv2.imread('road_image.jpg')
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

In [ ]:
# 장애물 추가 (사각형 박스로 시뮬레이션)
image_with_obstacle = image.copy()
h, w = image.shape[:2]
ox, oy = w // 2 + 0, h // 2 + 100
cv2.rectangle(image_with_obstacle, (ox, oy), (ox + 200, oy + 200), (200, 200, 200), cv2.FILLED)

plt.imshow(cv2.cvtColor(image_with_obstacle, cv2.COLOR_BGR2RGB))

In [ ]:
# 전처리: 그레이스케일 변환 및 노이즈 제거
gray    = cv2.cvtColor(image_with_obstacle, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

plt.imshow(blurred, cmap="gray")

In [ ]:
# 소벨 엣지 검출 (수직 엣지 강조 - 차선 검출에 유리)
sobelx     = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
abs_sobelx = np.uint8(np.absolute(sobelx))

plt.imshow(abs_sobelx, cmap="gray")

In [ ]:
# [NumPy 직접 구현] 소벨 엣지 검출 — OpenCV 결과와 비교
def my_sobel_conv(img, kernel):
    """패딩 후 3×3 커널 합성곱 (경계: reflect)"""
    padded = np.pad(img.astype(np.float64), pad_width=1, mode="reflect")
    out = np.zeros_like(img, dtype=np.float64)
    for i in range(3):
        for j in range(3):
            out += kernel[i, j] * padded[i:i + img.shape[0], j:j + img.shape[1]]
    return out

In [ ]:
# 3×3 소벨 커널
Kx = np.array([[-1, 0, 1],
               [-2, 0, 2],
               [-1, 0, 1]], dtype=np.float64)

my_sobelx     = my_sobel_conv(blurred, Kx)
my_abs_sobelx = np.uint8(np.absolute(my_sobelx))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(abs_sobelx,    cmap="gray"); axes[0].set_title("cv2.Sobel Kx")
axes[1].imshow(my_abs_sobelx, cmap="gray"); axes[1].set_title("my_sobel Kx")
plt.tight_layout()
plt.show()

print("최대 픽셀 오차:", np.max(np.abs(abs_sobelx.astype(int) - my_abs_sobelx.astype(int))))

In [ ]:
# 소벨 엣지 검출 (수평 엣지 강조)
sobely     = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
abs_sobely = np.uint8(np.absolute(sobely))

plt.imshow(abs_sobely, cmap="gray")

In [ ]:
# 3×3 소벨 커널
Ky = np.array([[-1, -2, -1],
               [ 0,  0,  0],
               [ 1,  2,  1]], dtype=np.float64)

my_sobely     = my_sobel_conv(blurred, Ky)
my_abs_sobely = np.uint8(np.absolute(my_sobely))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(abs_sobely,    cmap="gray"); axes[0].set_title("cv2.Sobel Ky")
axes[1].imshow(my_abs_sobely, cmap="gray"); axes[1].set_title("my_sobel Ky")
plt.tight_layout()
plt.show()

print("최대 픽셀 오차:", np.max(np.abs(abs_sobely.astype(int) - my_abs_sobely.astype(int))))

In [ ]:
# 소벨 엣지 검출 (magnitude: X+Y 결합)
magnitude = np.sqrt(sobelx**2 + sobely**2)
magnitude = np.uint8(np.clip(magnitude, 0, 255))

plt.imshow(magnitude, cmap="gray")

### 2.3 Canny(캐니) 엣지 검출 — 4단계 알고리즘

- **노이즈에 강하고 가늘고 정확한 엣지를 찾기 위해 고안된 4단계 알고리즘**

1. **노이즈 제거**: 5×5 가우시안 필터 적용
2. **그래디언트 계산**: 소벨 커널로 각 픽셀의 강도 변화량과 방향 계산
3. **비최대 억제(Non-maximum Suppression)**: 그래디언트 방향에서 최대값이 아닌 픽셀 제거 → **엣지를 1픽셀 두께로 가늘게**
4. **이중 임계값(Hysteresis Thresholding)**:
    - `threshold2 (Max)`: 이 값보다 높은 강도 → 무조건 엣지
    - `threshold1 (Min)`: 이 값보다 낮은 강도 → 무조건 무시
    - 두 값 사이: 확실한 엣지와 연결된 경우에만 엣지로 채택
    - **권장 비율: 2:1 또는 3:1**

**참고**
https://www.educative.io/answers/what-is-canny-edge-detection

In [ ]:
# 전처리: 그레이스케일 변환 및 노이즈 제거
gray    = cv2.cvtColor(image_with_obstacle, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

In [ ]:
# 캐니 엣지 검출 (차선 및 장애물 경계선 추출)
edges = cv2.Canny(blurred, 50, 150, L2gradient=True)  # NumPy 구현 결과와 맞추기 위해 L2gradient=True 적용

plt.imshow(edges, cmap="gray")

In [ ]:
# 윤곽선 찾기 → 객체화
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
# 결과 시각화 (장애물에 사각형 그리기)
result = image_with_obstacle.copy()
for cnt in contours:
    if cv2.contourArea(cnt) > 900:          # 너무 작은 노이즈 제거
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(result, (x, y), (x+w, y+h), (0, 255, 0), 2)

plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))

In [ ]:
# [NumPy 직접 구현] Canny 4단계 + 컨투어 → 바운딩 박스

In [ ]:
# ── 1단계: 소벨 그래디언트 (크기 + 방향) ──────────────────────────────────
Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
Ky = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float64)

Gx = my_sobel_conv(blurred, Kx)
Gy = my_sobel_conv(blurred, Ky)

G     = np.sqrt(Gx**2 + Gy**2)
theta = np.arctan2(Gy, Gx) * 180 / np.pi  # -180 ~ 180 도(degree)
theta = (theta + 180) % 180                # 0 ~ 180 으로 정규화

In [ ]:
# ── 2단계: 비최대 억제 (Non-Maximum Suppression) ──────────────────────────
# 그래디언트 방향으로 이웃 두 픽셀보다 크지 않으면 0으로 제거 → 엣지를 1픽셀 두께로
H, W = G.shape
nms = np.zeros_like(G)

for y in range(1, H - 1):
    for x in range(1, W - 1):
        ang = theta[y, x]
        g   = G[y, x]

        # 방향을 4개 축(0°, 45°, 90°, 135°)으로 양자화
        if (ang < 22.5) or (ang >= 157.5):            # 수평 방향
            n1, n2 = G[y, x - 1], G[y, x + 1]
        elif ang < 67.5:                               # 45° 대각선
            n1, n2 = G[y - 1, x - 1], G[y + 1, x + 1]
        elif ang < 112.5:                              # 수직 방향
            n1, n2 = G[y - 1, x], G[y + 1, x]
        else:                                          # 135° 대각선
            n1, n2 = G[y - 1, x + 1], G[y + 1, x - 1]

        # 동일 크기 그래디언트로 인한 2픽셀 두께 현상을 방지하기 위해 한쪽은 > 로 검사
        if g >= n1 and g > n2:
            nms[y, x] = g

In [ ]:
# ── 3단계: 이중 임계값 (Hysteresis Thresholding) ──────────────────────────
low, high = 50, 150
strong  = nms >= high
weak    = (nms >= low) & (nms < high)
edges_np = np.zeros_like(G, dtype=np.uint8)
edges_np[strong] = 255

# 약한 엣지 중 강한 엣지와 8-연결된 것만 살림 (반복 탐색을 통해 모두 추적)
changed = True
while changed:
    changed = False
    for y in range(1, H - 1):
        for x in range(1, W - 1):
            if weak[y, x] and edges_np[y, x] == 0:
                if (edges_np[y-1:y+2, x-1:x+2] == 255).any():
                    edges_np[y, x] = 255
                    changed = True

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(edges,    cmap="gray"); axes[0].set_title("OpenCV Canny Edge")
axes[1].imshow(edges_np, cmap="gray"); axes[1].set_title("NumPy Canny Edge")
plt.tight_layout()
plt.show()

In [ ]:
# 윤곽선 찾기 → 객체화
contours, _ = cv2.findContours(edges_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

In [ ]:
# 결과 시각화 (장애물에 사각형 그리기)
result = image_with_obstacle.copy()
for cnt in contours:
    if cv2.contourArea(cnt) > 900:          # 너무 작은 노이즈 제거
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(result, (x, y), (x+w, y+h), (0, 255, 0), 2)

plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))

### 2.4 Laplacian(라플라시안) 엣지 검출 개요

- **이미지의 2차 미분**을 사용하여 밝기 변화의 속도가 변하는 지점(Zero-crossing)을 탐지
- 엣지의 방향성을 따지지 않고 **모든 방향의 급격한 변화를 한꺼번에 감지**
- **주의**: 2차 미분 방식이므로 작은 노이즈도 크게 증폭됨 → **반드시 가우시안 블러 전처리 필요**

| 알고리즘 | 미분 차수 | 노이즈 강인성 | 엣지 두께 |
|---------|-----------|-------------|---------|
| Sobel | 1차 | 낮음 | 두꺼움 |
| Laplacian | 2차 | 매우 낮음 | 얇음 |
| **Canny** | **1차 + 후처리** | **높음** | **1픽셀** |

**참고**
https://docs.opencv.org/3.4/d5/db5/tutorial_laplace_operator.html

In [ ]:
# Laplacian
laplacian = cv2.Laplacian(blurred, cv2.CV_64F)
abs_laplacian = np.uint8(np.absolute(laplacian))

plt.imshow(cv2.cvtColor(abs_laplacian, cv2.COLOR_BGR2RGB))

### 2.5 (TODO) Canny 임계값 튜닝 실험

`cv2.Canny()`의 `threshold1`과 `threshold2`를 다양하게 바꿔보며
엣지 검출 결과가 어떻게 달라지는지 관찰하세요.

In [ ]:
# TODO: 여기에 구현하세요
# 힌트:
#   - blurred 이미지를 사용합니다.
#   - (30, 90) / (50, 150) / (100, 300) 세 가지 임계값 조합을 각각 적용하세요.
#   - plt.imshow()로 결과를 나란히 출력하고 엣지 수와 두께 변화를 비교하세요.
#   - 권장 비율(2:1 또는 3:1)을 지키며 실험해보세요.

### 2.6 실제 로봇 환경에서의 주의사항

**실제 주행 환경은 실험실과 달리 매우 불안정**
- **조명 변화**: HSV 색 공간의 V(Value) 채널 활용 또는 적응형 임계값 처리로 극복
- **그림자**: 색상 정보를 분리하여 처리하는 것이 중요
- **노이즈 영향**: **블러링(Blurring) 전처리는 필수**

**엣지 검출 시 흔히 겪는 문제 3가지**
1. **전처리 생략** — 블러 처리를 하지 않아 이미지 전체가 노이즈 엣지로 뒤덮이는 경우
2. **컬러 이미지 바로 사용** — 반드시 **그레이스케일로 변환 후 처리**
3. **고정 임계값 고집** — 환경에 맞춰 임계값을 동적으로 조절하거나 최적값을 찾아야 함

---
## Part 3: 퀴즈 & 복습

### 3.1 코드 읽기 문제

**Q1. 다음 코드에서 `(5, 5)`가 의미하는 것은?**
```python
blurred = cv2.GaussianBlur(img, (5, 5), 0)
```

**Q2. 물체에서 '수직 방향 경계선'을 강조하려면 어느 코드가 맞는가?**
```python
# A: cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
# B: cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
```

**Q3. 다음 코드 실행 순서가 올바른가?**
```python
1. edges   = cv2.Canny(gray, 100, 200)
2. blurred = cv2.GaussianBlur(edges, (5, 5), 0)
```

### 3.2 (TODO) 종합 실습: 나만의 엣지 검출 파이프라인 만들기

아래 요구사항을 만족하는 로봇 비전 파이프라인을 직접 구현하세요.

**요구사항**
1. 임의의 컬러 이미지(또는 직접 생성한 테스트 이미지)를 불러온다.
2. 그레이스케일로 변환한다.
3. 적절한 블러 필터로 노이즈를 제거한다.
4. Canny 엣지 검출을 적용한다.
5. 검출된 윤곽선 중 면적이 큰 것(>?00)만 녹색 사각형으로 표시한다.
6. 원본·그레이·블러·엣지·검출 결과를 각각 창으로 출력한다.

In [ ]:
# TODO: 여기에 구현하세요
# 힌트:
#   - cv2.imread(), cv2.cvtColor(), cv2.GaussianBlur(), cv2.Canny()
#   - cv2.findContours(), cv2.contourArea(), cv2.boundingRect(), cv2.rectangle()
#   - plt.imshow()